In [ ]:
#!/usr/bin/env python3
"""
Transmit BPSK with:
- Binary mapping
- Upsampling by sps
- RRC pulse shaping
via UHD on a USRP B210.
"""

import uhd
import time
import numpy as np
import scipy.signal as signal

#########################
# Configurable Settings #
#########################
TX_RATE = 500e3    # Symbol rate * sps => actual sample rate to USRP
FREQ      = 2.45e9  # Must match TX
FREQ = uhd.libpyuhd.types.tune_request(FREQ)
GAIN    = 30.0     # TX gain (dB)
SPS     = 4        # Samples per symbol (upsampling factor)
ALPHA   = 0.35     # RRC roll-off factor
NUM_TAPS= 101      # RRC filter length
#######################################################

def rrc_filter(num_taps, alpha, sps):
    """
    Generate an RRC (Root Raised Cosine) filter impulse response.
    """
    # This implementation is simplistic; there are more robust ways to generate RRC.
    t = np.arange(-num_taps//2, num_taps//2 + 1, dtype=np.float64) / sps
    h = []
    for ti in t:
        if ti == 0.0:
            val = 1.0 - alpha + 4*alpha/np.pi
        else:
            numerator   = (np.sin(np.pi*ti*(1-alpha)) +
                           4*alpha*ti*np.cos(np.pi*ti*(1+alpha)))
            denominator = np.pi*ti * (1 - (4*alpha*ti)**2)
            if abs(denominator) < 1e-10:
                val = 1.0 - alpha + 4*alpha/np.pi  # approximate
            else:
                val = numerator / denominator
        h.append(val)
    h = np.array(h, dtype=np.float64)
    # Normalize filter energy
    h /= np.sqrt(np.sum(h*h))
    return h.astype(np.float32)

def string_to_bits(msg: str):
    """
    Convert string to bits (MSB first per byte).
    """
    bits = []
    for c in msg:
        val = ord(c)
        for i in reversed(range(8)):
            bits.append((val >> i) & 0x01)
    return np.array(bits, dtype=np.uint8)

def bits_to_bpsk_symbols(bits):
    """
    Map bits -> BPSK: 0 -> +1, 1 -> -1.
    """
    # Real axis only
    return np.where(bits == 0, 1.0, -1.0).astype(np.float32)

def upsample_and_filter(symbols, rrc, sps):
    """
    Upsample BPSK symbols by sps and apply RRC filter.
    Return complex float array (imag part=0).
    """
    # Upsample
    upsampled = np.zeros(len(symbols)*sps, dtype=np.float32)
    upsampled[::sps] = symbols

    # Filter with RRC (FIR)
    shaped = signal.lfilter(rrc, 1.0, upsampled)

    # Convert to complex
    return shaped.astype(np.complex64)

def main():
    #-----------------------------
    # 1. USRP Setup
    #-----------------------------
    usrp = uhd.usrp.MultiUSRP()
    
    # The actual sample rate to the USRP is symbol_rate * sps.
    # If TX_RATE=500e3, and sps=4, your symbol rate is 125 ksym/s.
    usrp.set_tx_rate(TX_RATE)
    print(f"TX rate set to {usrp.get_tx_rate()} Hz")

    usrp.set_tx_freq(FREQ,0)
    usrp.set_tx_gain(GAIN,0)
    usrp.set_tx_antenna("TX/RX",0)
    usrp.set_tx_bandwidth(TX_RATE,0)

    #-----------------------------
    # 2. Generate BPSK signal
    #-----------------------------
    # Add a known preamble for frame sync (e.g., a short Barker sequence or just some pattern).
    # We'll pick a short pattern, e.g., 13 bits from a Barker code: 1110001000101
    preamble_bits = np.array([1,1,1,0,0,0,1,0,0,0,1,0,1], dtype=np.uint8)
    
    # We'll send a text message
    message = "Hello, USRP with BPSK!"
    payload_bits = string_to_bits(message)

    # Combine preamble + payload
    tx_bits = np.concatenate([preamble_bits, payload_bits])

    # Map to BPSK
    bpsk_symbols = bits_to_bpsk_symbols(tx_bits)  # real-valued
    
    # Generate RRC filter
    rrc = rrc_filter(NUM_TAPS, ALPHA, SPS)

    # Upsample & filter
    tx_signal = upsample_and_filter(bpsk_symbols, rrc, SPS)

    # Optionally repeat the signal multiple times
    # so there's a continuous burst.
    # repeats = 10
    # tx_buffer = np.tile(tx_signal, repeats)

    # #-----------------------------
    # # 3. Create UHD streamer
    # #-----------------------------
    # stream_args = uhd.usrp.StreamArgs("fc32", "sc16")
    # tx_streamer = usrp.get_tx_stream(stream_args)

    # #-----------------------------
    # # 4. Transmit
    # #-----------------------------
    # print("[TX] Starting transmission...")
    # md = uhd.types.TXMetadata()
    # md.start_of_burst = True
    # md.end_of_burst   = False
    # md.has_time_spec  = False  # transmit ASAP

    # num_sent = tx_streamer.send(tx_buffer, len(tx_buffer), md)
    # if num_sent < len(tx_buffer):
    #     print(f"[WARNING] Only sent {num_sent}/{len(tx_buffer)} samples!")

    # # End burst
    # md.start_of_burst = False
    # md.end_of_burst   = True
    # tx_streamer.send(np.array([], dtype=np.complex64), 0, md)

    while True:
        # for center_freq in center_freqs:
        center_freqs = 2.45e9 # Hz
        usrp.send_waveform(tx_signal,.1, center_freqs, TX_RATE, [0], GAIN)
    print("[TX] Transmission complete!")
    time.sleep(0.1)

if __name__ == "__main__":
    main()


[INFO] [UHD] linux; GNU C++ version 13.3.0; Boost_108300; UHD_4.7.0.0-149-g635ad362
[INFO] [B200] Detected Device: B210
[INFO] [B200] Operating over USB 3.
[INFO] [B200] Initialize CODEC control...
[INFO] [B200] Initialize Radio control...
[INFO] [B200] Performing register loopback test... 
[INFO] [B200] Register loopback test passed
[INFO] [B200] Performing register loopback test... 
[INFO] [B200] Register loopback test passed
[INFO] [B200] Setting master clock rate selection to 'automatic'.
[INFO] [B200] Asking for clock rate 16.000000 MHz... 
[INFO] [B200] Actually got clock rate 16.000000 MHz.
[INFO] [B200] Asking for clock rate 32.000000 MHz... 
[INFO] [B200] Actually got clock rate 32.000000 MHz.


TX rate set to 500000.0 Hz
